In [1]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import (
    DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR,
    MODEL_OUTPUT_DIR, RESULTS_DIR
)

In [2]:
df = pd.read_csv(PROCESSED_DATA_DIR / "cleaned_combined.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(2)

Shape: (121084, 11)
Columns: ['label', 'text', 'num_urls', 'num_exclamation', 'num_question', 'num_dollar', 'num_all_caps', 'num_numbers', 'word_count', 'capital_ratio', 'emoji_count']


,label,text,num_urls,num_exclamation,num_question,num_dollar,num_all_caps,num_numbers,word_count,capital_ratio,emoji_count
0,0,receiv nahou-msmbx01v [phonenumber] nahou-msmb...,41,0,0,3,240,407,2319,0.145347,0
1,0,receiv nahou-msmbx03vcorpenroncom [phonenumber...,0,1,0,0,19,56,166,0.230851,0


In [3]:
df["label"].value_counts()

label
1    61083
0    60001
Name: count, dtype: int64

In [4]:
TEXT_COL = "text"
TARGET = "label"

NUM_COLS = [
    "num_urls", "num_exclamation", "num_question", "num_dollar",
    "num_all_caps", "num_numbers", "word_count", "capital_ratio",
    "emoji_count"
]

X_text = df[TEXT_COL]
X_num = df[NUM_COLS]
y = df[TARGET]

print(f"Text samples: {len(X_text)}")
print(f"Numeric features: {len(NUM_COLS)}")
print(f"Target distribution:\n{y.value_counts()}")

Text samples: 121084
Numeric features: 9
Target distribution:
label
1    61083
0    60001
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = (
    train_test_split(X_text, X_num, y, test_size=0.2, random_state=42, stratify=y)
)

print(f"Train: {len(y_train)}  Test: {len(y_test)}")
print(f"Train pos: {y_train.sum()}  Test pos: {y_test.sum()}")

Train: 96867  Test: 24217
Train pos: 48866  Test pos: 12217


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=30000, sublinear_tf=True, ngram_range=(1,2), max_df=0.7, min_df=3)

X_text_train_tfidf = tfidf.fit_transform(X_text_train)
X_text_test_tfidf = tfidf.transform(X_text_test)

print(f"TF-IDF train shape: {X_text_train_tfidf.shape}")
print(f"TF-IDF test shape:  {X_text_test_tfidf.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

X_train = hstack([X_text_train_tfidf, X_num_train_scaled])
X_test = hstack([X_text_test_tfidf, X_num_test_scaled])

print(f"Combined train shape: {X_train.shape}")
print(f"Combined test shape:  {X_test.shape}")

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC(class_weight="balanced", random_state=42, max_iter=2000, dual='auto')
svm.fit(X_train, y_train)

train_acc = svm.score(X_train, y_train)
test_acc = svm.score(X_test, y_test)
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

y_pred = svm.predict(X_test)
y_score = svm.decision_function(X_test)

print("=== SVM Results ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_score):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Ham", "Spam"],
            yticklabels=["Ham", "Spam"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - SVM (Combined)")
plt.tight_layout()
plt.show()

In [ ]:
feature_names = tfidf.get_feature_names_out().tolist() + NUM_COLS
coefs = svm.coef_.flatten()

top_spam_idx = np.argsort(coefs)[-20:]
top_ham_idx = np.argsort(coefs)[:20]

print("Top 20 features for SPAM:")
for i in reversed(top_spam_idx):
    print(f"  {feature_names[i]:30s} {coefs[i]:+.4f}")

print("\nTop 20 features for HAM:")
for i in top_ham_idx:
    print(f"  {feature_names[i]:30s} {coefs[i]:+.4f}")

In [ ]:
import emoji
import re
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
from src.preprocess import TextPreprocessor

def clean_db_text(text):
    tp = TextPreprocessor(text)
    tp.strip_html_tags()
    tp.text = re.sub(r'[\[\]]', ' ', tp.text)
    tp.text = re.sub(r'http\S+|www\.\S+', ' [URL] ', tp.text)
    tp.replace_emails()
    tp.replace_phone_numbers()
    tp.replace_numbers()
    tp.replace_percentages()
    tp.replace_emojis()
    tp.remove_special_characters()
    tp.normalize_whitespace()
    tp.to_lowercase()
    return tp.get_text()

In [ ]:
def extract_features(email):
    db = pd.DataFrame()
    db["num_urls"]        = email.str.findall(r'https?://\S+|www\.\S+').str.len()
    db["num_exclamation"] = email.str.count(r'!')
    db["num_question"]    = email.str.count(r'\?')
    db["num_dollar"]      = email.str.count(r'\$')
    db["num_all_caps"]    = email.str.findall(r'\b[A-Z]{2,}\b').str.len()
    db["num_numbers"]     = email.str.findall(r'\d+').str.len()
    db["word_count"]      = email.str.split().str.len()
    caps   = email.str.findall(r'[A-Z]').str.len()
    letters = email.str.findall(r'[A-Za-z]').str.len()
    db["capital_ratio"]   = np.where(letters > 0, caps / letters, 0)
    db["emoji_count"]     = email.apply(emoji.emoji_count)
    db["text"] = email.apply(clean_db_text)
    ps = PorterStemmer()
    stop_words = set(stopwords.words("english"))
    db["text"] = db["text"].apply(lambda t: " ".join(
        ps.stem(w) for w in t.split()
        if w.lower() not in stop_words and w not in string.punctuation
    ))
    db["text"] = db["text"].str.replace(r'[(),;]', ' ', regex=True)
    db["text"] = db["text"].str.replace(r'\s+', ' ', regex=True).str.strip()
    return db

In [ ]:
from src.preprocess import parse_eml

email2 = parse_eml(PROCESSED_DATA_DIR / "email-testing-5.eml")[1]
sample_email = pd.Series([email2])
sample_features = extract_features(sample_email)

# test the model
sample_tfidf = tfidf.transform(sample_features["text"])
sample_num = scaler.transform(sample_features[NUM_COLS])
sample_combined = hstack([sample_tfidf, sample_num])
sample_pred = svm.predict(sample_combined)
sample_score = svm.decision_function(sample_combined)[0]

print(f"number of emojis: {sample_features['emoji_count'].values[0]}")
print(f"Sample email prediction: {'SPAM' if sample_pred[0] == 1 else 'HAM'}")
print(f"Sample email decision value: {sample_score:.4f}")

In [ ]:
test_emails = [
    "Congratulations! You won $1000 free cash prize. Click here now to claim!!!",
    "Hi John, please find attached the meeting notes from yesterday's call.",
    "URGENT: Your account has been suspended. Verify now at http://scam-site.com",
    "Can we reschedule tomorrow's lunch to 1pm instead?",
    "Buy cheap v1agra 100% free shipping limited offer act now!!!",
    "For your Assignment 1, word count of 3100 is already slightly too high. If you still want to add more content, you may need to remove some existing content.It depends on how you present your ideas and how much those ideas are 'borrowed' from various sources. I suggest you provide citations to those sources that have strong influence on your ideas.For Assignment 2A, you can assume that all edges in the map have positive weight. Hence, there is no need to worry about negative weight. Using another language for GUI (I supposed you are referring to using game engine + scripting) is allowed as long it can be invoked directly from your main program and does not require complex setup. Make sure you explain clearly how to execute your program in the report.",
    "Please be reminded that the Assignment 1 report is due this Sunday (17th May). No further extension is allowed (except under special circumstances) as the submission deadline has been extended due to Canvas disruption. Note that you need to adhere to the word count (2500 - 3000 words). Report that is too long or too short will be penalised.Report that is slightly above or below the limit (within 10%) is accepted without penaltyEach picture/diagram is equivalent to 50 words.Submit early so that if the Turnitin similarity score is too high (>24%), you still have time to improve it. Turnitin similarity score of 25% - 40% is subjected to penalty and if the score is higher than 40%, a plagiarism hearing might be required.",
    "Show, Last weekend, I was sick and couldn't do much of anything. So I used Claude Code to build a simple game. Think Frogger meets Space Invaders (it was actually pretty fun... DM me if you want to check it out).Of course, I expected the usual issues, like bugs and rough edges — but this wasn't the biggest development hurdle. The code was often correct. The problem was drift.Before I explain how I tackled this challenge: here's your reminder that AI April is nearly over. There's only few days left to take advantage of our best discount of the year:",
    "Supporters can now pay you in USDC, and the money lands in your account in your local currency. Same payouts, same dashboard, nothing new for you to learn.It's already on for your page. Next time a supporter wants to pay this way, they can.Many thanks,Buy Me a Coffee Team PS: If you're on Stripe Standard, you'll need to enable this from your settings before your supporters can pay with USDC.",
    "This email was sent to: SHOWWAIYAN555@GMAIL.COM You have received this email because you are a Maybank customer. To avoid future emails from Maybank being automatically sent to your junk mail folder, we suggest that you add our email address m2u@maybank.com.my to your Address Book and/or the Approved Sender list.DisclaimerYou are advised not to send any confidential and/or important information to the Bank via e-mailThe Personal Data Protection Act 2010 has been enforced on 15 December 2013. Please read our Privacy Notice which can be found at https://www.maybank.com/en/privacy-statement.page or at any branch for details on how we process/protect your personal data.This email and any of its attachments are confidential and may also be privileged. If you are not the intended recipient, please disregard the email and delete it immediately. Do not disclose, copy, circulate or in any other way, use or rely on the information contained in this email or its attachments. Email are not guaranteed to be secure or error-free as the message and any of its content could be intercepted, corrupted, lost, delayed, incomplete and amended. Please do not reply to this email.If you do not wish to receive any promotional information from Maybank via email, unsubscribe here.",
        '''
Payment Success for [ 26052270538 ]
Inbox

Fiuu <notification@fiuu.com>
Fri, May 22, 2:11 PM (1 day ago)
to me

Merchant Logo	Payment successful
MYR 12.20
PAYMENT DETAILS
Transaction ID	3722933396
Email	au****@gmail.com
Contact No.	10***6487
Service Items	ZUS Coffee billing
Payment Method	Credit/Debit Card
Order Number	26052270538
Name of Cardholder	Aun***********hu
Authorized Date/Time	2026-05-22 14:10:55
Approval code	154170
support support@fiuu.com	
Powered by   fiuu
    '''
]

df_test = extract_features(pd.Series(test_emails))
X_test_tfidf = tfidf.transform(df_test["text"])
X_test_num = scaler.transform(df_test[NUM_COLS])
X_test_combined = hstack([X_test_tfidf, X_test_num])
y_test_pred = svm.predict(X_test_combined)
y_test_scores = svm.decision_function(X_test_combined)
for i, (email, pred, score) in enumerate(zip(test_emails, y_test_pred, y_test_scores)):
    label = "SPAM" if pred == 1 else "HAM"
    print(f"[{i}] {label} (decision: {score:.4f}) — {email[:70]}...")


In [ ]:
joblib.dump(tfidf, MODEL_OUTPUT_DIR / "tfidf_50000_combined_svm.pkl")
joblib.dump(svm, MODEL_OUTPUT_DIR / "svm_combined.pkl")
joblib.dump(scaler, MODEL_OUTPUT_DIR / "scaler.pkl")
print(f"Saved TF-IDF vectorizer to {MODEL_OUTPUT_DIR / 'tfidf_50000_combined_svm.pkl'}")
print(f"Saved model to {MODEL_OUTPUT_DIR / 'svm_combined.pkl'}")
print(f"Saved scaler to {MODEL_OUTPUT_DIR / 'scaler.pkl'}")